In [2]:
import yfinance as yf
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
import pandas as pd

os.makedirs("data", exist_ok=True)

ModuleNotFoundError: No module named 'yfinance'

In [ ]:
START = "2018-01-01"
END = "2025-12-31"
TRADING_DAYS = 252

tickers = {
    "S&P500": "^GSPC",
    "EUROSTOXX": "^STOXX50E",
    "EUR_BOND": "^TNX",  
    "EURUSD": "EURUSD=X",
    "GOLD": "GC=F"
}

weighting = {
    "Stocks": 0.4, 
    "Bonds": 0.3, 
    "Currencies": 0.2, 
    "Commodities": 0.1}


In [ ]:
data = yf.download(list(tickers.values()), start=START, end=END, auto_adjust=False)["Close"]
data = data.rename(columns={ticker: name for name, ticker in tickers.items()})
data = data[list(tickers.keys())]

data = data.ffill() # if nan, fill with previous value
#print(data.isna().sum())
data = data.dropna() #remove rows with missing values
print(data.tail())
data.describe();

# plot the price evolution of each asset
plt.figure(figsize=(12, 6))
for col in data.columns:
    plt.plot(data.index, data[col], label=col)
plt.title("Price evolution of assets")
plt.xlabel("Date")
plt.ylabel("Price")
plt.legend()
plt.show()


In [ ]:
# cumulative from simple returns
simple_returns = data.pct_change().dropna() # simple returns
print("Daily returns:")
print(simple_returns.head())

cum_assets_simple = (1 + simple_returns).cumprod()
cum_assets_simple.plot(figsize=(12, 6))
plt.title("Cumulative performance based on daily simple returns")
plt.xlabel("Date")
plt.ylabel("Cumulative performance")
plt.legend()
plt.show()


# cumulative from daily log returns
log_returns = np.log(data / data.shift(1)).dropna() # log returns
print("Daily log returns:")
print(log_returns.head())

cum_assets_log = np.exp(log_returns.cumsum())
cum_assets_log.plot(figsize=(12, 6))
plt.title("Cumulative performance based on daily log returns")
plt.xlabel("Date")
plt.ylabel("Cumulative performance")
plt.legend()
plt.show()


In [ ]:
# correlation matrices based on log returns
corr_pearson = log_returns.corr()
corr_spearman = log_returns.corr(method="spearman")

# heatmap Pearson
plt.figure(figsize=(8, 6))
sns.heatmap(corr_pearson, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Pearson correlation (log returns)")
plt.show()

In [ ]:
portfolio = {
    'Stocks': ['S&P500', 'EUROSTOXX'],
    'Bonds': ['EUR_BOND'],
    'Currencies': ['EURUSD'],
    'Commodities': ['GOLD']
}

weights = {}
for cls, assets in portfolio.items():
    for asset in assets:
        weights[asset] = weighting[cls] / len(assets)
weights = pd.Series(weights) # convertion in serties for easier manipulation
#print("Poids par actif :\n", weights)
#print("Total poids:", weights.sum())

# daily portfolio returns and cumulative portfolio performance
#print(log_returns)
#print(log_returns[weights.index])
#print(log_returns * weights)

#print(weights.sum())
port_returns = (log_returns[weights.index] * weights).sum(axis=1) # daily portfolio returns
port_cum = np.exp(port_returns.cumsum()) #@ cumulative portfolio performance
#print(port_cum)

plt.figure(figsize=(12, 6))
port_cum.plot(label='Portfolio', linewidth=2)
plt.title("Cumulative portfolio performance")
plt.xlabel("Date")
plt.ylabel("Cumulative performance")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
# save data and results
data.to_csv("data/prices.csv")
log_returns.to_csv("data/returns_log.csv")
port_returns.to_frame("Portfolio").to_csv("data/portfolio_returns_log.csv")
weights.to_csv("data/weights.csv",  index=True)

print("Saved")
